# UMT-RSIAT resume on Kaggle
Notebook này tiếp tục full run CIFAR-100 seed 1993 từ checkpoint `task_N.pkl`. Trước khi chạy: bật GPU và Internet trong Notebook Settings; thêm checkpoint Colab hoặc output Kaggle trước đó bằng **Add Input**. Không dùng checkpoint có prefix `smoke`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/PhThuan-tech/RSIAT.git'
BRANCH = 'main'
SEED = 1993
TASKS_PER_RUN = 1  # Tăng lên 2 chỉ khi bạn chắc thời lượng phiên GPU đủ.

WORKING_ROOT = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
PROJECT_DIR = WORKING_ROOT / 'RSIAT'
OUTPUT_ROOT = WORKING_ROOT / 'RSIAT_runs'
IMPORT_ROOT = WORKING_ROOT / 'imported_resume'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
IMPORT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/kaggle/tmp/huggingface'
os.environ['TORCH_HOME'] = '/kaggle/tmp/torch'

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())
print('Outputs:', OUTPUT_ROOT)
print('Tasks this session:', TASKS_PER_RUN)

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
assert torch.cuda.is_available(), 'Hãy bật GPU trong Kaggle Notebook Settings.'

In [ ]:
%pip install -q -r requirements.txt

## Tìm checkpoint resume
Notebook chấp nhận `task_N.pkl` trực tiếp hoặc file `rsiat_resume_seed1993.zip` trong `/kaggle/input`. Nếu có nhiều checkpoint, task lớn nhất được chọn.

In [ ]:
import re
import zipfile

for archive in INPUT_ROOT.rglob('rsiat_resume*.zip'):
    destination = IMPORT_ROOT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as stream:
        stream.extractall(destination)
    print('Extracted:', archive)

def task_number(path):
    match = re.search(r'task_(\d+)\.pkl$', path.name)
    return int(match.group(1)) if match else -1

checkpoint_candidates = [
    path for root in (INPUT_ROOT, IMPORT_ROOT)
    for path in root.rglob('task_*.pkl')
    if 'smoke' not in str(path).lower()
]
checkpoint_candidates.sort(key=task_number)
RESUME_CHECKPOINT = checkpoint_candidates[-1] if checkpoint_candidates else None

if RESUME_CHECKPOINT is None:
    print('Không tìm thấy checkpoint full; run sẽ bắt đầu từ task 0.')
else:
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location='cpu', weights_only=False)
    print('Resume checkpoint:', RESUME_CHECKPOINT)
    print('Completed task:', checkpoint['cur_task'])
    print('Known classes:', checkpoint['known_classes'])
    if 'class_order' in checkpoint:
        print('Class order head:', checkpoint['class_order'][:10])

## Chuẩn bị CIFAR-100
Nếu bạn đã thêm một Kaggle Dataset chứa thư mục `cifar-100-python`, notebook dùng nó ở chế độ chỉ đọc. Nếu không, torchvision tải CIFAR vào `/kaggle/working/RSIAT/data/datasets` (cần bật Internet).

In [ ]:
from torchvision.datasets import CIFAR100

local_data_root = PROJECT_DIR / 'data' / 'datasets'
cifar_inputs = list(INPUT_ROOT.rglob('cifar-100-python'))
if cifar_inputs:
    attached_root = cifar_inputs[0].parent
    if local_data_root.exists() and not local_data_root.is_symlink():
        shutil.rmtree(local_data_root)
    if not local_data_root.exists():
        local_data_root.symlink_to(attached_root, target_is_directory=True)
    print('Using attached CIFAR:', attached_root)
else:
    local_data_root.mkdir(parents=True, exist_ok=True)
    print('No attached CIFAR dataset; torchvision may download it once for this session.')

cifar_train = CIFAR100(root=str(local_data_root), train=True, download=not bool(cifar_inputs))
cifar_test = CIFAR100(root=str(local_data_root), train=False, download=not bool(cifar_inputs))
print('CIFAR train/test:', len(cifar_train), len(cifar_test))

In [ ]:
import json

config = json.loads(Path('exps/umt_adapter_cifar224.json').read_text())
config['seed'] = [SEED]
config['resume'] = True
config['output_root'] = str(OUTPUT_ROOT)
config['max_tasks_per_run'] = TASKS_PER_RUN
config['num_workers'] = 2
config['stats_num_workers'] = 2
if RESUME_CHECKPOINT is not None:
    resume_target_dir = (
        OUTPUT_ROOT / 'ckpt' / config['prefix'] / config['dataset']
        / f"{config['init_cls']}_{config['increment']}" / f'seed_{SEED}'
    )
    resume_target_dir.mkdir(parents=True, exist_ok=True)
    local_resume = resume_target_dir / RESUME_CHECKPOINT.name
    if not local_resume.exists():
        shutil.copy2(RESUME_CHECKPOINT, local_resume)
    config['resume_path'] = str(local_resume)
else:
    config.pop('resume_path', None)

KAGGLE_CONFIG = WORKING_ROOT / 'umt_kaggle.json'
KAGGLE_CONFIG.write_text(json.dumps(config, indent=2))
print('Config:', KAGGLE_CONFIG)
print('seed/resume/max_tasks:', config['seed'], config['resume'], config['max_tasks_per_run'])
print('resume_path:', config.get('resume_path'))

In [ ]:
!python -m unittest tests.test_research_components
!python main.py --config /kaggle/working/umt_kaggle.json

## Đóng gói checkpoint cho phiên sau
Chỉ chạy cell này sau khi trainer kết thúc sạch và đã in `Saved checkpoint`.

In [ ]:
import json
import shutil

output_checkpoints = sorted(OUTPUT_ROOT.rglob('task_*.pkl'), key=task_number)
assert output_checkpoints, 'Không có checkpoint output mới.'
latest = output_checkpoints[-1]
manifest = {
    'seed': SEED,
    'latest_checkpoint': str(latest.relative_to(OUTPUT_ROOT)),
    'completed_task': task_number(latest),
}
(OUTPUT_ROOT / 'resume_manifest.json').write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive(
    str(WORKING_ROOT / f'rsiat_resume_seed{SEED}_task{task_number(latest)}'),
    'zip',
    root_dir=OUTPUT_ROOT,
)
print('Latest checkpoint:', latest)
print('Resume bundle:', archive)
print('Hãy tải ZIP này hoặc lưu nó thành Kaggle Dataset/Notebook Output để dùng cho phiên sau.')

## Phiên Kaggle tiếp theo
1. Tải `rsiat_resume_seed1993_taskN.zip` xuống hoặc tạo private Dataset từ output. 2. Mở phiên mới và Add Input bundle đó. 3. Chạy notebook từ đầu; nó tự chọn task lớn nhất và tiếp tục task N+1. Nếu bật **Persistence: Files**, `/kaggle/working` có thể được giữ giữa interactive sessions, nhưng vẫn nên giữ bundle riêng vì persistence là best-effort.